In [ ]:
#package
suppressMessages({
  library(Seurat)
  library(tidyverse)
  library(tidyr)
  library(ggplot2)
  library(patchwork)
  library(dplyr)
  library(cowplot)
  library(future)
  library(DoubletFinder)
  library(RColorBrewer)
  library(harmony)
  library(R.utils)  
library(data.table)
library(statmod)
library(limma)
library(pheatmap)
library(clusterProfiler)
#library(GEOquery)
library(ggsci)
library(ggvenn)
})
#volcano plot
p = ggplot("your_document_path",aes(x=as.numeric(avg_log2FC),y= -log10(p_val_adj),color=group0.1)) +
    geom_point(alpha=0.65,size=1)+#size and opacity regulation
     guides(
    color = guide_legend(override.aes = list(size=5))  # dot size in legend
  )+
    scale_color_manual(values=c('#546de5','#dedae2','#ff4757'))+xlim(c(-1.5,1.5))+#dot color
    geom_vline(xintercept=c(-cut_off_log2FC,cut_off_log2FC),lty=4,col="black",lwd=0.8)+#type of line:'twodash','longdash','dotdash','dotted','dashed','solid','blank'
    geom_hline(yintercept= -log10(cut_off_padj),lty=4,col='black',lwd=0.8)+
    labs(x=bquote(~Log[2]~ "fold change"),y= bquote(~-Log[10]~ "adjusted p-value"))+
    #ggtitle("")+#title
    theme_bw()+#theme
    theme(plot.title=element_text(hjust = 0.5),
         legend.position="right",
         legend.title = element_blank(),
          legend.key.size=unit(2,'cm'),
          legend.text=element_text(size=20),
         axis.title=element_text(size=20),
          axis.text=element_text(size=15)
          )+NoLegend()
# GObarplot
mytheme = theme(axis.title=element_text(size=20),
              axis.text=element_text(size=16),
              axis.text.y=element_blank(),
              axis.ticks.length.y=unit(0,"cm"),
              plot.title=element_text(size=25,hjust=0.5,face="bold"),
              legend.title=element_text(size=18),
              legend.text=element_text(size=15),
                plot.margin=margin(t=10,r=20,l=10,b=10)
)
p=ggplot(data="your_document_path",aes(x=-log10(p.adjust),y=rev(Description))) +
geom_bar(stat='identity',width=0.5,fill='#a0d9f6',alpha=0.85)+
scale_x_continuous(
  expand = expansion(mult = c(0, 0))#,  # right extension
  #limits = c(0, NA)  # NA auto_compucate the max
)+
scale_y_discrete(expand = expansion(add = c(1, 1)))+
labs(x=expression(-log[10]("Adjusted p-value")),
    y='GO Pathway',
    title='GO Pathway Enrichment')+
geom_text(size=10,aes(x=0.05,label=Description),hjust=0)+
geom_text(size=10,aes(x=0.05,label=geneID),hjust=0,vjust=3.0,color='#a0d9f6')+
theme_classic()+NoLegend()+
mytheme
#expession plot
library(broom)
raw.df <- FetchData("seuratobject", vars = c("your_goal_genes", "group")) %>%
  tibble::rownames_to_column("cell") %>%
  tidyr::pivot_longer(
    cols      = all_of("your_goal_genes"),
    names_to  = "gene",
    values_to  = "value"
  )
pvals <- raw.df %>%
  group_by(gene) %>%
  do(tidy(wilcox.test(value ~ group, data = .))) %>%
  mutate(
    p.signif = case_when(
      p.value < 0.0001 ~ "****",
      p.value < 0.001  ~ "***",
      p.value < 0.01   ~ "**",
      p.value < 0.05   ~ "*",
      TRUE             ~ "ns"
    )
  )
y.max <- raw.df %>%
  group_by(gene, group) %>%
  summarise(
    mean = mean(value),
    se   = sd(value) / sqrt(n()),
    .groups = "drop"
  ) %>%
  group_by(gene) %>%
  summarise(y.pos = max(mean + se) * 1.15)
pvals <- left_join(pvals, y.max, by = "gene")
p <- ggplot(raw.df, aes(x = gene, y = value, fill = group)) +

  # 柱状图（均值）
  stat_summary(
    fun   = mean,
    geom  = "bar",
    position = position_dodge(width = 0.7),
    width = 0.6,
    color = "black",
    linewidth = 0.4
  ) +

  # 误差棒（SEM）
  stat_summary(
    fun.data = mean_se,
    geom     = "errorbar",
    position = position_dodge(width = 0.7),
    width    = 0.2,
    linewidth = 0.5
  ) +

  # 显著性标记（手动）
  geom_text(
    data = pvals,
    aes(x = gene, y = y.pos, label = p.signif),
    inherit.aes = FALSE,
    size = 5
  ) +

  # 配色
  scale_fill_manual(values = c("Control" = "#7BAFD4", "HS" = "#C8706A")) +

  # Y 轴
  scale_y_continuous(expand = expansion(mult = c(0, 0.25))) +

  # 标题
  labs(
    title = "Expression of Selected Genes",
    x     = NULL,
    y     = "Mean Expression"
  ) +

  # 主题
  theme_classic(base_size = 11) +
  theme(
    plot.title   = element_text(face = "bold", hjust = 0.5, size = 12),
    axis.text.x  = element_text(angle = 45, hjust = 1, size = 20),
    axis.text.y  = element_text(color = "black", size = 20),
    axis.title.y = element_text(size = 20),
    legend.position = "right"
  )
#Veen plot
library(VennDiagram)
row_df1 <- rownames(dataA)
row_df2 <- rownames(dataB)

data_list <- list(
  "dataA" = row_df1,
  "dataB" = row_df2
    )
p=ggvenn(data_list,stroke_size=0.1,show_percentage = TRUE,
  digits = 1,
  fill_color = c("#acdbdf", "#d7eaea"),
  fill_alpha = 0.8,
  stroke_color = "gray30",
  stroke_alpha = 0.8,
  stroke_linetype = "solid",
  set_name_size = 4,
  text_color = "black",
  text_size = 5.2)
# p1=pC+ annotate("text", 
#            x = 0,  
#            y = 0.2,  
#            label = "dataC", 
#            size = 6, 
          
#            color = "black")
#heatmap
library(viridis)
library(RColorBrewer)
library(dplyr)
heatmap_matrix=as.matrix(your_heatmap_data_exp)
heatmap_zscore=t(scale(t(heatmap_matrix)))
heatmap_zscore[is.infinite(heatmap_zscore)]=0
heatmap_zscore[is.na(heatmap_zscore)]=0
#sample_metadata <- data.frame(Group = group_list,  row.names = colnames(heatmap_data))
if(!all(colnames(heatmap_data_exp_symbol) %in% rownames(sample_info))) {stop("check the data")}
sample_info=sample_info[colnames(heatmap_data_exp), , drop = FALSE]
# create_advanced_heatmap <- function(expression_matrix,
#                                    annotation_df,
#                                    cluster_rows = TRUE,
#                                    cluster_cols = TRUE,
#                                    show_rownames = FALSE,
#                                    show_colnames = TRUE,
#                                    fontsize_row = 6,
#                                    fontsize_col = 8,
#                                   ) {
expression_matrix=heatmap_zscore
annotation_df=sample_info 
  main_title = "DEGs Expression Heatmap"
  color_palette <- colorRampPalette(c("blue", "white", "red"))(3467
  group_colors <- list(
    Group = c("TLE_HS" = "#E41A1C", "ControlCortex" = "#377EB")
  
  pheatmap::pheatmap(expression_matrix,
           color = color_palette,
           scale = "none",
           # cluster_rows = cluster_rows,
           cluster_cols = F,
           # clustering_distance_rows = "euclidean",
           # clustering_distance_cols = "euclidean",
           # clustering_method = "complete",
           annotation_col = annotation_df,
           annotation_colors = group_colors,
            show_colnames = F,
           filename = "heatmap.pdf",  
           width = 12,               
            height = 16 ,
             border_color = NA        )
           # show_rownames = show_rownames,
           # show_colnames = show_colnames,
           # fontsize_row = fontsize_row,
           # fontsize_col = fontsize_col,
           # border_color = NA,
           # main = main_title,
           # legend = TRUE,
           # treeheight_row = 30,
           # treeheight_col = 30,
           # cellwidth = ifelse(show_colnames, 8, NA),
           # cellheight = ifelse(show_rownames, 6, NA),
           # silent = TRUE)  # silent = TRUE 